# CELL 1 - Load data from github

In [0]:
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Load từ GitHub
url = "https://raw.githubusercontent.com/lethanhconghcmus-alt/retail-mlops/main/data/processed/clean_transactions.parquet"

# Pandas đọc parquet từ URL
pdf = pd.read_parquet(url)

# Convert sang Spark DataFrame
df = spark.createDataFrame(pdf)

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Rows: 805,549
Columns: 9
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- total_price: double (nullable = true)



# Cell 2 - Customer Aggregation với PySpark


In [0]:
from pyspark.sql import functions as F

# Reference date để tính recency
reference_date = df.select(F.max("InvoiceDate")).collect()[0][0]

customer_features = df.groupBy("Customer ID").agg(
    # Monetary
    F.round(F.sum("total_price"), 2).alias("total_spent"),
    F.round(F.avg("Price"), 2).alias("avg_unit_price"),

    # Frequency
    F.countDistinct("Invoice").alias("num_invoices"),
    F.count("*").alias("num_line_items"),

    # Time-based
    F.min("InvoiceDate").alias("first_purchase"),
    F.max("InvoiceDate").alias("last_purchase"),

    # Diversity
    F.countDistinct("StockCode").alias("num_unique_products"),
    F.countDistinct("Country").alias("num_countries"),
)

# Tính recency và derived features
customer_features = customer_features.withColumn(
    "recency_days",
    F.datediff(F.lit(reference_date), F.col("last_purchase"))
).withColumn(
    "active_days",
    F.datediff(F.col("last_purchase"), F.col("first_purchase")) + 1
).withColumn(
    "purchase_frequency",
    F.round(F.col("num_invoices") / F.col("active_days"), 4)
).withColumn(
    "avg_basket_value",
    F.round(F.col("total_spent") / F.col("num_invoices"), 2)
)

print(f"Unique customers: {customer_features.count():,}")
customer_features.printSchema()
customer_features.show(5, truncate=False)

Unique customers: 5,878
root
 |-- Customer ID: string (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- avg_unit_price: double (nullable = true)
 |-- num_invoices: long (nullable = false)
 |-- num_line_items: long (nullable = false)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- num_unique_products: long (nullable = false)
 |-- num_countries: long (nullable = false)
 |-- recency_days: integer (nullable = true)
 |-- active_days: integer (nullable = true)
 |-- purchase_frequency: double (nullable = true)
 |-- avg_basket_value: double (nullable = true)

+-----------+-----------+--------------+------------+--------------+-------------------+-------------------+-------------------+-------------+------------+-----------+------------------+----------------+
|Customer ID|total_spent|avg_unit_price|num_invoices|num_line_items|first_purchase     |last_purchase      |num_unique_products|num_countries|recency_days|active_days|p

## PySpark Customer Aggregation

Từ **805,549 transaction rows** → aggregate xuống còn **5,878 customer rows**.
Mỗi customer được đại diện bởi 12 features:

### Feature Groups:

**Monetary:**
- `total_spent` - tổng chi tiêu toàn bộ thời gian
- `avg_unit_price` - giá trung bình sản phẩm mua → proxy cho taste/segment
- `avg_basket_value` - giá trị trung bình mỗi lần mua

**Frequency:**
- `num_invoices` - số lần mua hàng (unique orders)
- `num_line_items` - tổng số dòng sản phẩm
- `purchase_frequency` - invoices/ngày → đo độ trung thành

**Recency:**
- `recency_days` - số ngày kể từ lần mua cuối → customer còn active không?
- `active_days` - khoảng thời gian từ lần đầu đến lần cuối mua

**Diversity:**
- `num_unique_products` - độ đa dạng sản phẩm
- `num_countries` - số quốc gia mua hàng (>1 có thể là reseller)

> Framework này chính là **RFM (Recency - Frequency - Monetary)**
> mở rộng thêm diversity features - standard approach trong
> retail customer segmentation.

# Cell 3 - Save customer features

In [0]:
# Cell 3 — Save customer features
customer_features.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/retail_data/customer_features.parquet"
)
print("Saved!")

Saved!
